# QC-Only Lot & Bag Gap Check

**Purpose:** Standalone check — separate from the main merging notebook.

Scans `from_qcAnalyst/` and `raw_spectro/` and flags any lot or bag that QC
recorded but Spectro never physically measured.

**Three scenarios detected:**
1. A lot missing entirely from Spectro, sitting inside a range of lots Spectro
   *did* actively test (e.g. Spectro has 8212AM, 8213AM, 8215AM — 8214AM is
   missing and gets flagged if QC has it).
2. A lot Spectro measured, but with a bag or bag-range missing compared to
   what QC recorded for that lot.
3. A lot QC only recorded as part of a range (e.g. QC has 8212-8213 as one
   range entry) that falls short of a lot Spectro actually tested (e.g.
   Spectro has 8212, 8213, 8214 — 8214 is flagged since QC's range doesn't
   reach it).

**Output:** one Excel file, one sheet, four columns only —
`Lot Number`, `Bag`, `Product Code`, `Oversize`.

**Note:** This notebook intentionally does NOT run the full QC/Spectro
validation & correction pipeline from `data_merging_version-3.ipynb` (typo
fixes, product code corrections, etc.). It assumes QC and Spectro files are
already in their normal expected formats. If a lot/bag format doesn't parse,
it's printed as a flagged/skipped item rather than silently dropped or
guessed at — bring those to Ms. Jam like usual.

In [12]:
# ---- Imports ----
import pandas as pd
import re
import os
import glob
from datetime import datetime
from zoneinfo import ZoneInfo
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

pd.set_option("display.max_rows", 200)

## 1. Load QC file (from `from_qcAnalyst/`)

In [13]:
qc_folder = "from_qcAnalyst"
qc_files = glob.glob(os.path.join(qc_folder, "*.csv"))

if len(qc_files) == 0:
    raise FileNotFoundError(f"No QC CSV found in '{qc_folder}/'.")
elif len(qc_files) > 1:
    raise ValueError(f"More than one QC file found in '{qc_folder}/' — expected only one:\n" + "\n".join(qc_files))

qc_path = qc_files[0]
qc_df = pd.read_csv(qc_path)

# Drop the blank spacer rows the QC export inserts between real records
qc_df = qc_df.dropna(subset=["id"]).reset_index(drop=True)

print(f"QC file loaded: {qc_path}")
print(f"Rows (after dropping blank spacers): {len(qc_df)}")

QC file loaded: from_qcAnalyst\qc_report_20260701.csv
Rows (after dropping blank spacers): 12553


## 2. Load Spectro files (from `raw_spectro/`)

In [14]:
spectro_folder = "raw_spectro"
spectro_files = glob.glob(os.path.join(spectro_folder, "*.xlsx"))

if len(spectro_files) == 0:
    raise FileNotFoundError(f"No Spectro .xlsx files found in '{spectro_folder}/'.")

spectro_data = {}  # product_code -> dataframe

for f in spectro_files:
    filename = os.path.basename(f)
    name_no_ext = os.path.splitext(filename)[0]
    product_code = name_no_ext.split(" ")[0]  # text before first space
    df = pd.read_excel(f)
    df = df.dropna(how="all").reset_index(drop=True)  # drop fully-blank separator rows
    spectro_data[product_code] = df
    print(f" - {product_code}: {len(df)} rows, file = '{filename}'")

print(f"\nTotal Spectro files loaded: {len(spectro_data)}")

 - YA16164E: 401 rows, file = 'YA16164E.xlsx'
 - YA16192E: 70 rows, file = 'YA16192E.xlsx'

Total Spectro files loaded: 2


## 3. Normalize QC columns

Uppercase/trim text fields, fix bag numbers mangled into date-like text
(e.g. `"Jan-40"` → `"1-40"`), and recover bag numbers from remarks when the
dedicated field is blank — same logic as the main notebook.

In [15]:
def clean_column(column):
    return column.where(
        column.isna(),
        column.astype(str).str.replace(r"\s+", "", regex=True).str.upper()
    )

str_cols = ["lot_number", "product_code", "remarks", "internal_lot", "bag_no"]
for c in str_cols:
    if c in qc_df.columns:
        qc_df[c] = clean_column(qc_df[c])

MONTH_MAP = {
    'JAN': 1, 'FEB': 2, 'MAR': 3, 'APR': 4, 'MAY': 5, 'JUN': 6,
    'JUL': 7, 'AUG': 8, 'SEP': 9, 'OCT': 10, 'NOV': 11, 'DEC': 12
}

BAG_NUMBER_PATTERN = re.compile(r'^\d{1,3}(?:-\d{1,3})?$')


def fix_bag_no(value):
    """
    Normalize bag numbers to one of these formats:

        0
        00
        000
        0-0
        00-00
        000-000
        0-00
        0-000

    Leading zeros are preserved.
    """

    if pd.isna(value) or str(value).strip() == '':
        return None
    value = str(value).strip().upper()
    # Convert month names that Excel may have introduced.
    def replace_month(match):
        return str(MONTH_MAP[match.group(0).upper()])
    value = re.sub(
        r'(?i)(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)',
        replace_month,
        value
    )
    # Remove spaces.
    value = re.sub(r'\s+', '', value)
    # Normalize common separators to '-'.
    value = re.sub(r'[/\\–—_]+', '-', value)
    # Keep only digits and '-'.
    value = re.sub(r'[^0-9-]', '', value)
    # Remove duplicate hyphens.
    value = re.sub(r'-+', '-', value)
    # Remove hyphens from the beginning/end.
    value = value.strip('-')
    if not value:
        return None
    # Handle a range.
    if '-' in value:
        parts = value.split('-')
        if len(parts) != 2:
            return None
        start, end = parts
        if not start or not end:
            return None
        # Each side must contain 1-3 digits.
        if not (1 <= len(start) <= 3 and 1 <= len(end) <= 3):
            return None
        # Normalize range direction while preserving zero padding.
        if int(start) > int(end):
            start, end = end, start
        return f'{start}-{end}'

    # Handle a single bag number.
    if value.isdigit() and 1 <= len(value) <= 3:
        return value

    # Cannot safely determine how an oversized number should be split.
    return None

qc_df["bag_no_corrected"] = qc_df["bag_no"].apply(fix_bag_no)

# ---- Recover bag numbers from remarks when bag_no is blank ----
BAG_REMARKS_PATTERN = re.compile(r'\bbag\b\s*#?\s*(\d{1,3}(?:\s*-\s*\d{1,3})?)', re.IGNORECASE)

def normalize_bag_spacing(remarks_value):
    if pd.isna(remarks_value):
        return remarks_value
    text = str(remarks_value)
    text = re.sub(r'(?i)\b(mega|maga|meg)\s+bag\b', 'megabag', text)
    text = re.sub(r'(?i)\bbag\s*#\s*', 'bag#', text)
    text = re.sub(r'(?i)#\s+', '#', text)
    return text

def extract_bag_from_remarks(remarks_value):
    if pd.isna(remarks_value):
        return None
    match = BAG_REMARKS_PATTERN.search(str(remarks_value))
    if match:
        return match.group(1).replace(' ', '')
    return None

qc_df["remarks_normalized"] = qc_df["remarks"].apply(normalize_bag_spacing)
needs_fallback = qc_df["bag_no_corrected"].isna() | (qc_df["bag_no_corrected"].astype(str).str.strip() == '')
qc_df.loc[needs_fallback, "bag_no_corrected"] = qc_df.loc[needs_fallback, "remarks_normalized"].apply(extract_bag_from_remarks)

print(f"QC rows: {len(qc_df)}")
print(f"Bag numbers recovered from remarks: {(needs_fallback & qc_df['bag_no_corrected'].notna()).sum()}")

QC rows: 12553
Bag numbers recovered from remarks: 46


## 4. Determine matching lot key per QC row

Mode B clients (PACKAGEWORLD/ABI, ROWELL, DYNAMICCAPS/NICE — 9 product
codes) are matched by internal lot number instead of sticker lot number,
same as the main notebook.

In [16]:
MODE_B_CODES = {"BA12556E", "WA12282E", "WA15151E", "WA15816E",
                "BA17042E", "BA17070E",
                "WA7997E", "WA15218E", "WA15229E"}

def get_internal_lot_key(row):
    internal_lot = row.get("internal_lot")
    if pd.notna(internal_lot) and str(internal_lot).strip() != "":
        return str(internal_lot).strip().upper()
    lot_number = str(row["lot_number"]).strip()
    match = re.match(r"^.*?\(([A-Za-z0-9]+)\)$", lot_number)
    if match:
        inner = match.group(1).strip()
        if re.search(r"[A-Za-z]", inner):
            return inner.upper()
    return None

OS_REMARKS_PATTERN = re.compile(r"\b(from\s+os|os)\b", re.IGNORECASE)

def check_os_in_remarks(remarks_value):
    if pd.isna(remarks_value):
        return False
    return bool(OS_REMARKS_PATTERN.search(str(remarks_value)))

print("Mode key logic ready.")

Mode key logic ready.


## 5. Classify & expand QC lot numbers

Handles: single lot, lot range, `LOT(bag)` parenthesis, `LOT(bag-bag)`
parenthesis range. Anything else is flagged and skipped (printed below) so
it isn't silently mishandled.

In [17]:
def classify_qc_lot(value):
    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return "blank"
    if "(" in value and ")" in value:
        return "parenthesis"
    if "-" in value:
        return "range"
    return "single"

def split_lot_parts(lot):
    match = re.match(r"^(\d+)([A-Za-z]+)$", lot.strip())
    if not match:
        return None, None, None
    return match.group(1), match.group(2), len(match.group(1))

def next_letters(letters):
    letters = list(letters.upper())
    i = len(letters) - 1
    while i >= 0:
        if letters[i] != 'Z':
            letters[i] = chr(ord(letters[i]) + 1)
            break
        else:
            letters[i] = 'A'
            i -= 1
    return "".join(letters)

def expand_range(start_lot, end_lot):
    start_num_str, start_letters, num_len = split_lot_parts(start_lot)
    end_num_str, end_letters, _ = split_lot_parts(end_lot)
    if start_num_str is None or end_num_str is None:
        return None
    result = []
    current_num = int(start_num_str)
    current_letters = start_letters
    max_value = 10 ** num_len - 1
    safety_counter, max_iterations = 0, 5000
    while True:
        current_lot = f"{str(current_num).zfill(num_len)}{current_letters}"
        result.append(current_lot)
        if current_lot == end_lot.strip().upper():
            break
        current_num += 1
        if current_num > max_value:
            current_num = 1
            current_letters = next_letters(current_letters)
        safety_counter += 1
        if safety_counter > max_iterations:
            print(f"WARNING: range {start_lot}-{end_lot} exceeded safety limit, stopped.")
            break
    return result

# ---- Build flat QC list: one entry per (product_code, mode, lot, bag_range, is_oversize) ----
# `range_group` tags every lot that came from the same QC range-format row (e.g.
# "8212AM-8217AM"), so scenario 3 can check the group as a whole: if Spectro
# tested ANY lot from that group, every untested lot in that same group is
# flagged too — even if it's far from the tested lots by itself.
qc_flat = []
skipped_qc_rows = []
range_group_counter = 0

for idx, row in qc_df.iterrows():
    product_code = str(row["product_code"]).strip()
    is_mode_b = product_code in MODE_B_CODES
    mode = "B" if is_mode_b else "A"
    lot_value = str(row["lot_number"]).strip()
    lot_format = classify_qc_lot(lot_value)
    bag_val = row.get("bag_no_corrected")
    is_os = check_os_in_remarks(row.get("remarks"))
    qc_id = row.get("id")

    if is_mode_b:
        key = get_internal_lot_key(row)
        lot_key = key if key else lot_value.upper()
        qc_flat.append({"product_code": product_code, "mode": mode, "lot": lot_key,
                         "bag_raw": bag_val, "is_oversize": is_os, "range_group": None})
        continue

    # Mode A
    if lot_format == "single":
        qc_flat.append({"product_code": product_code, "mode": mode, "lot": lot_value.upper(),
                         "bag_raw": bag_val, "is_oversize": is_os, "range_group": None})

    elif lot_format == "range":
        parts = lot_value.split("-")
        if len(parts) == 2:
            expanded = expand_range(parts[0].strip(), parts[1].strip())
            if expanded:
                range_group_counter += 1
                group_id = range_group_counter
                for lot in expanded:
                    qc_flat.append({"product_code": product_code, "mode": mode, "lot": lot.upper(),
                                     "bag_raw": bag_val, "is_oversize": is_os, "range_group": group_id})
            else:
                skipped_qc_rows.append((qc_id, lot_value, "range failed to expand"))
        else:
            skipped_qc_rows.append((qc_id, lot_value, "unexpected range format"))

    elif lot_format == "parenthesis":
        m_single = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)\)$", lot_value)
        m_range = re.match(r"^(\d{4}[A-Za-z]{1,2})\((\d+)-(\d+)\)$", lot_value)
        if m_single:
            qc_flat.append({"product_code": product_code, "mode": mode, "lot": m_single.group(1).upper(),
                             "bag_raw": m_single.group(2), "is_oversize": is_os, "range_group": None})
        elif m_range:
            qc_flat.append({"product_code": product_code, "mode": mode, "lot": m_range.group(1).upper(),
                             "bag_raw": f"{m_range.group(2)}-{m_range.group(3)}", "is_oversize": is_os, "range_group": None})
        else:
            skipped_qc_rows.append((qc_id, lot_value, "unrecognized parenthesis format"))

    else:
        skipped_qc_rows.append((qc_id, lot_value, f"format={lot_format}"))

print(f"QC flat entries built: {len(qc_flat)}")
print(f"QC rows skipped (unparseable lot format): {len(skipped_qc_rows)}")
if skipped_qc_rows:
    for qid, val, reason in skipped_qc_rows[:30]:
        print(f"  QC id {qid}: '{val}' — {reason}")

QC flat entries built: 44889
QC rows skipped (unparseable lot format): 7
  QC id 6217: '7190AM-7192AM(15-19)' — unrecognized parenthesis format
  QC id 3192: '8761AL-8762AL(39-40)' — unrecognized parenthesis format
  QC id 2947: '8086AL-8087AL(40-41)' — unrecognized parenthesis format
  QC id 2865: '7839AL-7840AL(9-11)' — unrecognized parenthesis format
  QC id 2819: '7786AL-7788AL(17-24)' — unrecognized parenthesis format
  QC id 2705: '7306AL-7307AL(40-41)' — unrecognized parenthesis format
  QC id 1146: '4506AL--4522AL' — unexpected range format


## 6. Normalize Spectro Name column, split lot/bag, flag reference rows

In [18]:
def parse_bag_range(bag_val):
    if bag_val is None or pd.isna(bag_val) or str(bag_val).strip() == "":
        return None
    bag_val = str(bag_val).strip()
    if "-" in bag_val:
        try:
            a, b = bag_val.split("-")
            return int(a), int(b)
        except:
            return None
    try:
        n = int(bag_val)
        return n, n
    except:
        return None

def is_reference_row(value):
    if pd.isna(value):
        return True
    value = str(value).upper()
    return any(marker in value for marker in ["STD", "LIGHT", "DARK", "CMA", "%"])

def split_name_bag(value, is_ref):
    if is_ref:
        return value, None, False
    value = str(value).strip()
    is_os = value.upper().endswith(" OS")
    if is_os:
        value = value[:-3].strip()
    parts = value.split(" ", 1)
    lot_part = parts[0].strip()
    bag_part = parts[1].strip() if len(parts) > 1 else None
    if bag_part is not None and bag_part.upper() == "OS":
        bag_part, is_os = None, True
    return lot_part, bag_part, is_os

spectro_flat = []  # one entry per real (non-reference) Spectro reading

for code, df in spectro_data.items():
    name_col = "Name"
    df["_is_ref"] = df[name_col].apply(is_reference_row)
    for _, row in df[~df["_is_ref"]].iterrows():
        raw_name = str(row[name_col]).strip().upper()
        raw_name = re.sub(r"\s+", " ", raw_name)

        lot, bag, is_os = split_name_bag(raw_name, False)

        # Normalize Spectro bag number using the same rules as QC.
        bag = fix_bag_no(bag)

        spectro_flat.append({
            "product_code": code,
            "lot": lot.upper(),
            "bag_raw": bag,
            "is_oversize": is_os
        })

print(f"Spectro real (non-reference) readings: {len(spectro_flat)}")

Spectro real (non-reference) readings: 466


## 7. Scenario 1 & 3 — lots QC has but Spectro never measured

Two ways a missing lot gets flagged:
- **Scenario 3 (range-aware):** if a QC row was written as a lot *range*
  (e.g. `8212AM-8217AM`) and Spectro tested at least one lot from that exact
  range, every untested lot in that same range is flagged — even ones far
  from the tested lots (e.g. Spectro only reached 8212–8214, so 8215–8217
  are flagged too).
- **Scenario 1 (family clustering, for standalone single-lot QC rows):**
  reuses the main notebook's Block 6 approach — a lot is flagged only if it
  falls inside a cluster of lot numbers Spectro was *actively testing* (gap
  of 2 or less between tested lots of the same letter-suffix).

In [19]:
def parse_lot(lot):
    m = re.match(r"^(\d+)([A-Za-z]+)$", lot)
    if not m:
        return None
    return int(m.group(1)), m.group(2).upper()

missing_lot_flags = []  # scenario 1 & 3

spectro_lots_by_code = {}
for entry in spectro_flat:
    spectro_lots_by_code.setdefault(entry["product_code"], set()).add(entry["lot"])

qc_by_code = {}
for entry in qc_flat:
    qc_by_code.setdefault(entry["product_code"], []).append(entry)

for code, tested_lots in spectro_lots_by_code.items():
    parsed_spectro = [parse_lot(lot) for lot in tested_lots]
    parsed_spectro = [p for p in parsed_spectro if p]

    by_suffix = {}
    for num, suffix in parsed_spectro:
        by_suffix.setdefault(suffix, []).append(num)

    families = []  # (suffix, min_num, max_num)
    for suffix, nums in by_suffix.items():
        nums = sorted(set(nums))
        cluster = [nums[0]]
        for n in nums[1:]:
            if n - cluster[-1] <= 2:
                cluster.append(n)
            else:
                families.append((suffix, cluster[0], cluster[-1]))
                cluster = [n]
        families.append((suffix, cluster[0], cluster[-1]))

    def within_a_family(num, suffix):
        return any(s == suffix and lo <= num <= hi for s, lo, hi in families)

    # ---- Scenario 3: range-group aware ----
    # For every QC row that was a lot range, if Spectro tested ANY lot from
    # that same range, flag every lot in the range Spectro didn't test.
    group_entries = {}
    for qc_entry in qc_by_code.get(code, []):
        if qc_entry["range_group"] is not None:
            group_entries.setdefault(qc_entry["range_group"], []).append(qc_entry)

    seen_missing = set()
    flagged_group_ids = set()

    for group_id, entries in group_entries.items():
        group_lots = {e["lot"] for e in entries}
        if not (group_lots & tested_lots):
            continue  # no lot from this range was tested — not this scenario
        flagged_group_ids.add(group_id)
        for qc_entry in entries:
            lot = qc_entry["lot"]
            if lot in tested_lots or lot in seen_missing:
                continue
            seen_missing.add(lot)
            missing_lot_flags.append({
                "product_code": code,
                "lot": lot,
                "bag": qc_entry["bag_raw"] if qc_entry["bag_raw"] else "",
                "is_oversize": qc_entry["is_oversize"],
            })

    # ---- Scenario 1: family clustering, for standalone single-lot QC rows ----
    for qc_entry in qc_by_code.get(code, []):
        if qc_entry["range_group"] is not None:
            continue  # already handled by scenario 3 above
        lot = qc_entry["lot"]
        if lot in tested_lots or lot in seen_missing:
            continue
        parsed = parse_lot(lot)
        if not parsed or not within_a_family(*parsed):
            continue  # outside every actively-tested range — not flagged
        seen_missing.add(lot)
        missing_lot_flags.append({
            "product_code": code,
            "lot": lot,
            "bag": qc_entry["bag_raw"] if qc_entry["bag_raw"] else "",
            "is_oversize": qc_entry["is_oversize"],
        })

print(f"Lots flagged as missing entirely (scenario 1 & 3): {len(missing_lot_flags)}")
for f in missing_lot_flags:
    print(f"  [{f['product_code']}] {f['lot']} — Bag {f['bag'] or '(none given)'}")

Lots flagged as missing entirely (scenario 1 & 3): 8
  [YA16164E] 7651AM — Bag (none given)
  [YA16164E] 7252AM — Bag (none given)
  [YA16164E] 2706AM — Bag (none given)
  [YA16164E] 2717AM — Bag (none given)
  [YA16164E] 2083AM — Bag (none given)
  [YA16164E] 8467AL — Bag (none given)
  [YA16164E] 8457AL — Bag (none given)
  [YA16192E] 5955AM — Bag (none given)


## 8. Scenario 2 — lots Spectro measured, but with missing bags

For every lot present in both QC and Spectro, compares the full set of bag
numbers QC recorded against the set Spectro actually read, and reports any
missing bags as consecutive ranges.

In [20]:
def bag_range_to_set(bag_raw):
    r = parse_bag_range(bag_raw)
    if r is None:
        return set()
    return set(range(r[0], r[1] + 1))

def collapse_to_ranges(sorted_ints):
    ranges = []
    if not sorted_ints:
        return ranges
    start = prev = sorted_ints[0]
    for n in sorted_ints[1:]:
        if n == prev + 1:
            prev = n
        else:
            ranges.append((start, prev))
            start = prev = n
    ranges.append((start, prev))
    return ranges

missing_bag_flags = []  # scenario 2

# Bags Spectro actually read, per (product_code, lot)
spectro_bags_by_lot = {}
for entry in spectro_flat:
    key = (entry["product_code"], entry["lot"])
    spectro_bags_by_lot.setdefault(key, set()).update(bag_range_to_set(entry["bag_raw"]))

# Bags QC recorded, per (product_code, lot) — plus oversize flag per bag
qc_bags_by_lot = {}
qc_os_by_lot = {}
for entry in qc_flat:
    key = (entry["product_code"], entry["lot"])
    bags = bag_range_to_set(entry["bag_raw"])
    qc_bags_by_lot.setdefault(key, set()).update(bags)
    if entry["is_oversize"]:
        qc_os_by_lot.setdefault(key, set()).update(bags)

for key, qc_bags in qc_bags_by_lot.items():
    if not qc_bags:
        continue  # QC gave no bag number for this lot — nothing to compare
    product_code, lot = key
    if key not in spectro_bags_by_lot:
        continue  # lot not measured at all — already covered by scenario 1/3
    spectro_bags = spectro_bags_by_lot[key]
    missing_bags = sorted(qc_bags - spectro_bags)
    if not missing_bags:
        continue
    os_bags = qc_os_by_lot.get(key, set())
    for lo, hi in collapse_to_ranges(missing_bags):
        bag_str = str(lo) if lo == hi else f"{lo}-{hi}"
        is_os = any(b in os_bags for b in range(lo, hi + 1))
        missing_bag_flags.append({
            "product_code": product_code,
            "lot": lot,
            "bag": bag_str,
            "is_oversize": is_os,
        })

print(f"Missing bag ranges flagged (scenario 2): {len(missing_bag_flags)}")
for f in missing_bag_flags:
    print(f"  [{f['product_code']}] {f['lot']} — Bag {f['bag']}")

Missing bag ranges flagged (scenario 2): 0


## 9. Combine, dedupe, and build the output table

In [21]:
all_flags = missing_lot_flags + missing_bag_flags

output_df = pd.DataFrame(all_flags)
if not output_df.empty:
    output_df = output_df.rename(columns={
        "lot": "Lot Number",
        "bag": "Bag",
        "product_code": "Product Code",
        "is_oversize": "Oversize",
    })
    output_df = output_df[["Lot Number", "Bag", "Product Code", "Oversize"]]
    output_df = output_df.drop_duplicates().sort_values(by=["Product Code", "Lot Number", "Bag"]).reset_index(drop=True)
else:
    output_df = pd.DataFrame(columns=["Lot Number", "Bag", "Product Code", "Oversize"])

print(f"Total flagged rows: {len(output_df)}")
output_df

Total flagged rows: 8


,Lot Number,Bag,Product Code,Oversize
0,2083AM,,YA16164E,False
1,2706AM,,YA16164E,False
2,2717AM,,YA16164E,False
3,7252AM,,YA16164E,False
4,7651AM,,YA16164E,False
5,8457AL,,YA16164E,False
6,8467AL,,YA16164E,False
7,5955AM,,YA16192E,False


## 10. Export single Excel file

In [22]:
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

ph_now = datetime.now(ZoneInfo("Asia/Manila"))
filename = f"QC Only Gap Check - {ph_now.strftime('%B %d, %Y - %H-%M')}.xlsx"
output_path = os.path.join(output_folder, filename)

wb = Workbook()
ws = wb.active
ws.title = "QC_ONLY_GAPS"

HEADER_FILL = PatternFill(start_color="2F4F4F", end_color="2F4F4F", fill_type="solid")
HEADER_FONT = Font(color="FFFFFF", bold=True)
GREEN_FILL = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")

headers = ["Lot Number", "Bag", "Product Code", "Oversize"]
ws.append(headers)
for col_idx, _ in enumerate(headers, start=1):
    cell = ws.cell(row=1, column=col_idx)
    cell.fill = HEADER_FILL
    cell.font = HEADER_FONT
    cell.alignment = Alignment(horizontal="center")

for _, row in output_df.iterrows():
    ws.append([row["Lot Number"], row["Bag"], row["Product Code"], bool(row["Oversize"])])

for r in range(2, ws.max_row + 1):
    for c in range(1, 5):
        ws.cell(row=r, column=c).fill = GREEN_FILL

for col_idx, header in enumerate(headers, start=1):
    col_letter = get_column_letter(col_idx)
    max_len = max([len(str(header))] + [len(str(v)) for v in output_df[header]] if not output_df.empty else [len(header)])
    ws.column_dimensions[col_letter].width = max_len + 4

wb.save(output_path)
print(f"Saved: {output_path}")
print(f"Total rows: {len(output_df)}")

Saved: output\QC Only Gap Check - August 14, 2026 - 13-43.xlsx
Total rows: 8
